In [1]:
import ee
ee.Authenticate()
ee.Initialize()


Enter verification code:  4/1AcvDMrBIqXgJiON4X0Z2HTk9TRpqY9OxCJZZpv_3uXVz0mqK38FwdFxGcVI



Successfully saved authorization token.


In [8]:
import numpy as np
import ee
ee.Initialize()
def get_fire_state(date, region):
    fire_data = ee.ImageCollection('FIRMS').filterDate(date).mean().clip(region)
    fire_matrix = fire_data.select('T21').reduceRegion(ee.Reducer.toList(), region, scale=1000).get('T21').getInfo()
    return np.array(fire_matrix)

def get_elevation(region):
    elevation_data = ee.Image('USGS/SRTMGL1_003').clip(region)
    elevation_matrix = elevation_data.reduceRegion(ee.Reducer.toList(), region, scale=1000).get('elevation').getInfo()
    return np.array(elevation_matrix)

def get_wind(date, region):
    wind_data = ee.ImageCollection('NASA/NLDAS/FORA0125_H002').filterDate(date).mean().clip(region)
    wind_u = wind_data.select('wind_u').reduceRegion(ee.Reducer.toList(), region, scale=1000).get('wind_u').getInfo()
    wind_v = wind_data.select('wind_v').reduceRegion(ee.Reducer.toList(), region, scale=1000).get('wind_v').getInfo()
    return np.array(wind_u), np.array(wind_v)

def get_humidity(date, region):
    humidity_data = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD').filterDate(date).mean().clip(region)
    humidity_matrix = humidity_data.reduceRegion(ee.Reducer.toList(), region, scale=1000).get('precipitation').getInfo()
    return np.array(humidity_matrix)

def calculate_froude_number(wind_speed, flame_height):
    g = 9.81  # acceleration due to gravity in m/s^2
    froude_number = wind_speed / np.sqrt(g * flame_height)
    return froude_number
region = ee.Geometry.Rectangle([-121.9, 37.0, -121.6, 37.3])  # Example region
date = '2023-01-01'

fire_state = get_fire_state(date, region)
elevation = get_elevation(region)
wind_u, wind_v = get_wind(date, region)
humidity = get_humidity(date, region)

# Example flame height (this would ideally come from your dataset or another source)
flame_height = 10.0  # meters

# Calculate wind speed
wind_speed = np.sqrt(wind_u**2 + wind_v**2)

# Calculate Froude number
froude_number = calculate_froude_number(wind_speed, flame_height)

# Save data
np.save('data/fire_state.npy', fire_state)
np.save('data/elevation.npy', elevation)
np.save('data/wind_u.npy', wind_u)
np.save('data/wind_v.npy', wind_v)
np.save('data/humidity.npy', humidity)
np.save('data/froude_number.npy', froude_number)



In [9]:
import numpy as np

# Load the saved data
fire_state = np.load('data/fire_state.npy')
elevation = np.load('data/elevation.npy')
wind_u = np.load('data/wind_u.npy')
wind_v = np.load('data/wind_v.npy')
humidity = np.load('data/humidity.npy')
froude_number = np.load('data/froude_number.npy')

# Inspect the fire_state data
print("Fire State Data:\n", fire_state)
print("Shape of Fire State Data:", fire_state.shape)

# Inspect the elevation data
print("\nElevation Data:\n", elevation)
print("Shape of Elevation Data:", elevation.shape)

# Inspect the wind_u data
print("\nWind U Component Data:\n", wind_u)
print("Shape of Wind U Component Data:", wind_u.shape)

# Inspect the wind_v data
print("\nWind V Component Data:\n", wind_v)
print("Shape of Wind V Component Data:", wind_v.shape)

# Inspect the humidity data
print("\nHumidity Data:\n", humidity)
print("Shape of Humidity Data:", humidity.shape)

# Inspect the froude_number data
print("\nFroude Number Data:\n", froude_number)
print("Shape of Froude Number Data:", froude_number.shape)


Fire State Data:
 []
Shape of Fire State Data: (0,)

Elevation Data:
 [40 38 45 ... 78 77 71]
Shape of Elevation Data: (1089,)

Wind U Component Data:
 [3.47 3.47 3.47 ... 3.92 3.92 3.92]
Shape of Wind U Component Data: (1122,)

Wind V Component Data:
 [2.53 2.53 2.53 ... 3.46 3.46 3.46]
Shape of Wind V Component Data: (1122,)

Humidity Data:
 [104.43271637 104.43271637 104.43271637 ...  59.72641373  59.72641373
  59.72641373]
Shape of Humidity Data: (1122,)

Froude Number Data:
 [0.43357791 0.43357791 0.43357791 ... 0.5278966  0.5278966  0.5278966 ]
Shape of Froude Number Data: (1122,)
